# Lab 03 Homework

## 1. Data reading

Let's first see if every file is in the right place

In [2]:
from pathlib import Path

import polars as pl

DATA_DIR = Path("data")
monthly_files = [DATA_DIR / f"{month}.parquet" for month in range(1, 13)]
missing_files = [path for path in monthly_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Missing monthly Parquet files: "
        + ", ".join(str(path) for path in missing_files)
    )

zone_file = DATA_DIR / "taxi_zone_lookup.csv"
if not zone_file.is_file():
    raise FileNotFoundError(f"Missing taxi zone lookup file: {zone_file}")

Load the 2024 monthly taxi trip files and the taxi zone lookup as lazy Polars queries. Keep rides that start during 2024 and end no later than January 1, 2025

In [ ]:


rides_lf = pl.concat(
    [
        pl.scan_parquet(path).with_columns(
            pl.col("tpep_pickup_datetime").cast(pl.Datetime("ns")),
            pl.col("tpep_dropoff_datetime").cast(pl.Datetime("ns")),
        )
        for path in monthly_files
    ],
    how="vertical_relaxed",
)

pickup = pl.col("tpep_pickup_datetime")
dropoff = pl.col("tpep_dropoff_datetime")
rides_lf = (
    rides_lf
    .filter(
        (pickup >= pl.datetime(2024, 1, 1))
        & (pickup < pl.datetime(2025, 1, 1))
        & (dropoff <= pl.datetime(2025, 1, 1))
    )
)

zones_lf = (
    pl.scan_csv(zone_file)
)

Both objects are lazy. Let's their schemas and collect only a small preview

In [3]:
from IPython.display import display

print(rides_lf.collect_schema())
print(zones_lf.collect_schema())
display(rides_lf.head().collect())
display(zones_lf.head().collect())

Schema({'VendorID': Int16, 'tpep_pickup_datetime': Datetime(time_unit='ns', time_zone=None), 'tpep_dropoff_datetime': Datetime(time_unit='ns', time_zone=None), 'passenger_count': Int16, 'trip_distance': Float64, 'RatecodeID': Int16, 'store_and_fwd_flag': Categorical(ordering='physical'), 'PULocationID': Int16, 'DOLocationID': Int16, 'payment_type': Int16, 'fare_amount': Float64, 'extra': Float64, 'mta_tax': Float64, 'tip_amount': Float64, 'tolls_amount': Float64, 'improvement_surcharge': Float64, 'total_amount': Float64, 'congestion_surcharge': Float64, 'Airport_fee': Float64})
Schema({'LocationID': Int16, 'Borough': Categorical(ordering='physical'), 'Zone': Categorical(ordering='physical'), 'service_zone': Categorical(ordering='physical')})


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i16,datetime[ns],datetime[ns],i16,f64,i16,cat,i16,i16,i16,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,"""N""",186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""",140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""",236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""",79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""",211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0


LocationID,Borough,Zone,service_zone
i16,cat,cat,cat
1,"""EWR""","""Newark Airport""","""EWR"""
2,"""Queens""","""Jamaica Bay""","""Boro Zone"""
3,"""Bronx""","""Allerton/Pelham Gardens""","""Boro Zone"""
4,"""Manhattan""","""Alphabet City""","""Yellow Zone"""
5,"""Staten Island""","""Arden Heights""","""Boro Zone"""


## 2. Data cleaning and filtering

Clean the lazy trip query. Rate codes 1-6 and vendor IDs 1-2 are the recognized values, rate code 99 is the dictionary's null value, so it is excluded

In [4]:
money_columns = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
]
valid_vendor_ids = [1, 2]
valid_ratecode_ids = [1, 2, 3, 4, 5, 6]

trip_duration = (
    pl.col("tpep_dropoff_datetime")
    - pl.col("tpep_pickup_datetime")
)
over_1000 = pl.any_horizontal(
    *[
        (pl.col(column).abs() > 1000).fill_null(False)
        for column in money_columns
    ]
)

rides_lf = (
    rides_lf
    .with_columns(pl.col("passenger_count").fill_null(1))
    .filter(pl.col("passenger_count") != 0)
    .with_columns(
        pl.when(pl.col("passenger_count") > 6)
        .then(6)
        .otherwise(pl.col("passenger_count"))
        .alias("passenger_count")
    )
    .filter(trip_duration <= pl.duration(hours=2))
    .with_columns(
        [pl.col(column).abs().alias(column) for column in money_columns]
    )
    .filter(~over_1000)
    .filter(pl.col("VendorID").is_in(valid_vendor_ids))
    .filter(pl.col("RatecodeID").is_in(valid_ratecode_ids))
)

## 3. Data transformation

Group payment types into credit card, cash, and other; add borough names; and create airport-ride and weekday rush-hour flags.

In [5]:
pickup_time = pl.col("tpep_pickup_datetime").dt.time()
is_weekday = pl.col("tpep_pickup_datetime").dt.weekday() <= 5
is_morning_rush = pickup_time.is_between(
    pl.time(6, 30), pl.time(9, 30), closed="left"
)
is_evening_rush = pickup_time.is_between(
    pl.time(15, 30), pl.time(20, 0), closed="left"
)

pickup_boroughs = zones_lf.select(
    pl.col("LocationID").cast(pl.Int32).alias("PULocationID"),
    pl.col("Borough").alias("pickup_borough"),
)
dropoff_boroughs = zones_lf.select(
    pl.col("LocationID").cast(pl.Int32).alias("DOLocationID"),
    pl.col("Borough").alias("dropoff_borough"),
)

rides_lf = (
    rides_lf
    .with_columns(
        pl.when(pl.col("payment_type") == 1)
        .then(pl.lit("credit_card"))
        .when(pl.col("payment_type") == 2)
        .then(pl.lit("cash"))
        .otherwise(pl.lit("other"))
        .alias("payment_type"),
        (pl.col("Airport_fee").fill_null(0) != 0).alias("is_airport_ride"),
        (
            is_weekday & (is_morning_rush | is_evening_rush)
        ).fill_null(False).alias("is_rush_hour"),
    )
    .join(pickup_boroughs, on="PULocationID", how="left")
    .join(dropoff_boroughs, on="DOLocationID", how="left")
    .drop("PULocationID", "DOLocationID")
)

## 4. Feature extraction

Aggregate the cleaned rides into one row per pickup date. The one-hot categorical indicators are summed within each day, so each resulting column is a daily ride count for that category

In [6]:
borough_categories = (
    zones_lf
    .select(pl.col("Borough").drop_nulls().unique().sort())
    .collect()
    .get_column("Borough")
    .to_list()
)
if "Unknown" not in borough_categories:
    borough_categories.append("Unknown")

dummy_aggregations = []
for payment_type in ["credit_card", "cash", "other"]:
    dummy_aggregations.append(
        (pl.col("payment_type") == payment_type)
        .cast(pl.UInt32)
        .sum()
        .alias(f"payment_type_{payment_type}_rides")
    )

for borough in borough_categories:
    suffix = (
        borough.strip()
        .lower()
        .replace(" ", "_")
        .replace("/", "_")
    )
    dummy_aggregations.extend(
        [
            (
                (pl.col("pickup_borough").fill_null("Unknown") == borough)
                .cast(pl.UInt32)
                .sum()
                .alias(f"pickup_borough_{suffix}_rides")
            ),
            (
                (pl.col("dropoff_borough").fill_null("Unknown") == borough)
                .cast(pl.UInt32)
                .sum()
                .alias(f"dropoff_borough_{suffix}_rides")
            ),
        ]
    )

daily_features_lf = (
    rides_lf
    .with_columns(
        pl.col("tpep_pickup_datetime").dt.date().alias("date")
    )
    .group_by("date")
    .agg(
        pl.len().cast(pl.UInt32).alias("ride_count"),
        pl.col("is_airport_ride").cast(pl.UInt32).sum().alias("airport_ride_count"),
        pl.col("is_rush_hour").cast(pl.UInt32).sum().alias("rush_hour_ride_count"),
        pl.col("fare_amount").mean().alias("average_fare_amount"),
        pl.col("trip_distance").median().alias("median_trip_distance"),
        pl.col("total_amount").sum().alias("total_amount_sum"),
        pl.col("total_amount")
        .filter(pl.col("payment_type") == "credit_card")
        .sum()
        .alias("total_amount_paid_by_card"),
        pl.col("total_amount")
        .filter(pl.col("payment_type") == "cash")
        .sum()
        .alias("total_amount_paid_by_cash"),
        pl.col("total_amount")
        .filter(pl.col("payment_type") == "other")
        .sum()
        .alias("total_amount_paid_by_other"),
        pl.col("congestion_surcharge").sum().alias("congestion_surcharge_total"),
        pl.col("passenger_count").sum().alias("passenger_count_total"),
        *dummy_aggregations,
    )
    .with_columns(
        pl.col("date").dt.quarter().alias("quarter"),
        pl.col("date").dt.month().alias("month"),
        pl.col("date").dt.day().alias("day_of_month"),
        pl.col("date").dt.weekday().alias("day_of_week"),
        (pl.col("date").dt.weekday() > 5).alias("is_weekend"),
    )
    .sort("date")
)

dataset_path = DATA_DIR / "dataset.parquet"
dataset = daily_features_lf.collect()
dataset.write_parquet(dataset_path)
print(f"Saved {dataset.height} daily rows to {dataset_path}")

Saved 366 daily rows to data\dataset.parquet
